In [ ]:
from pathlib import Path
import json
import pandas as pd
from pynwb import NWBHDF5IO
from IPython.display import display, Markdown

repo = Path("/Users/euo9382/Documents/Repositories/analysis_Belal2026")
nwb_root = repo / "RNAscope data" / "NWB"

nwb_path = nwb_root / "L1.ST8" / "L1.ST8.nwb"

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


def pretty(value):
    if value is None:
        return "_null_"
    if isinstance(value, (list, tuple)):
        if not value:
            return "_[]_"
        return "\n".join(f"- {v}" for v in value)
    if isinstance(value, dict):
        if not value:
            return "_{}_"
        lines = []
        for k, v in value.items():
            lines.append(f"**{k}:** {pretty(v)}")
        return "\n".join(lines)
    return str(value)


def show_block(title, mapping):
    display(Markdown(f"## {title}"))
    if not mapping:
        display(Markdown("_No entries_"))
        return

    parts = []
    for k, v in mapping.items():
        parts.append(f"### {k}\n{pretty(v)}")

    display(Markdown("\n\n".join(parts)))


with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
    nwbfile = io.read()

    nwb_meta = {
        "session_description": nwbfile.session_description,
        "identifier": nwbfile.identifier,
        "session_start_time": nwbfile.session_start_time,
        "experiment_description": nwbfile.experiment_description,
        "experimenter": list(nwbfile.experimenter) if nwbfile.experimenter is not None else None,
        "lab": nwbfile.lab,
        "institution": nwbfile.institution,
        "protocol": nwbfile.protocol,
        "notes": nwbfile.notes,
        "keywords": list(nwbfile.keywords) if nwbfile.keywords is not None else None,
    }

    subject_meta = {}
    if nwbfile.subject is not None:
        subject_meta = {
            "subject_id": nwbfile.subject.subject_id,
            "description": nwbfile.subject.description,
            "species": nwbfile.subject.species,
            "sex": nwbfile.subject.sex,
            "genotype": nwbfile.subject.genotype,
            "age": nwbfile.subject.age,
            "strain": nwbfile.subject.strain,
        }

    custom_meta = None
    if "session_metadata_custom" in nwbfile.scratch:
        raw_custom = nwbfile.scratch["session_metadata_custom"].data
        if isinstance(raw_custom, bytes):
            raw_custom = raw_custom.decode()
        custom_meta = json.loads(raw_custom)

    counts = (
        nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
        .to_dataframe()
        .reset_index(drop=True)
    )

display(Markdown(f"# NWB Metadata: `{nwb_path.name}`"))
show_block("NWBFile", nwb_meta)
show_block("Subject", subject_meta)
show_block("Custom", custom_meta)

display(Markdown("## Full Experimenter Counts"))
display(
    counts.sort_values(
        ["condition", "cell_type", "session", "hemisphere", "field_index", "replicate"],
        kind="stable",
    ).reset_index(drop=True)
)